base code

In [1]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# model_name = "google/flan-t5-small"

# # Load tokenizer + model
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# # Prompt (important!)
# prompt = """Classify sentiment as Positive or Negative.
# Text: I love this product.
# Answer:"""

# # Tokenize
# inputs = tokenizer(prompt, return_tensors="pt")

# # Generate output
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=5,
#     do_sample=False
# )

# # Decode result
# result = tokenizer.decode(outputs[0], skip_special_tokens=True)

# print(result)

Initialization

In [9]:
# import libraries
from sklearn.metrics import f1_score
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
warnings.filterwarnings("ignore")

# set config
results_df = pd.DataFrame(columns=["prompt_template", "accuracy", "F1_score"])

In [26]:
# Load the dataset
train = pd.read_csv('data/6/train.csv')
train.drop(columns=['data_id'], inplace=True)
train_df = train.head(200).copy()

In [23]:
# data cleaning
def clean_text(df, text_col="text", target_col="sentiment"):

    df = df.dropna(subset=[text_col])
    df[text_col] = df[text_col].str.strip()
    df[text_col] = df[text_col].str[:2048]

    df[target_col] = df[target_col].str.lower()
    return df

# inference
def batch_predict(texts, prompt_template, tokenizer, model):
    prompts = [prompt_template.format(t=t) for t in texts]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

def pipeline(df, prompt_template):
    # data cleaning
    df = clean_text(df)

    # Load model
    model_name = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    # Set model to eval mode (faster)
    model.eval()

    # Apply in batches
    batch_size = 32
    preds = []

    for i in range(0, len(df), batch_size):
        batch_texts = df["text"].iloc[i:i+batch_size].tolist()
        preds.extend(batch_predict(batch_texts, prompt_template, tokenizer, model))

    df["predicted"] = [p.strip() for p in preds]

    # evaluation
    df['predicted'] = df['predicted'].str.lower()
    df['sentiment'] = df['sentiment'].str.lower()
    df['correct'] = df['sentiment'] == df['predicted']

    # print('Prompt Template:')
    # print('-'*10)
    # print(prompt_template)
    # print('-'*10)

    accuracy = df['correct'].mean()
    # print(f"Accuracy: {accuracy:.2%}")

    f1 = f1_score(df['sentiment'], df['predicted'], average='weighted')
    print(f"F1-Score: {f1}")

    print('-'*50)

    results = {
        "prompt_template": prompt_template,
        "accuracy": accuracy,
        "F1_score": f1
    }

    return results, df

Baseline

In [5]:
prompt_template = f"Classify sentiment as Positive or Negative.\nText: {{t}}\nAnswer:"
results, df = pipeline(valid_df, prompt_template)
results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
results_df

Prompt Template:
----------
Classify sentiment as Positive or Negative.
Text: {t}
Answer:
----------
Accuracy: 87.14%
F1-Score: 0.8792653061224489
--------------------------------------------------


/var/folders/xq/dznpw7s50vdg77ddlcz18rvw0000gn/T/ipykernel_25920/993365498.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)


,prompt_template,accuracy,F1_score
0,Classify sentiment as Positive or Negative.\nT...,0.871429,0.879265


Stronger zero-shot prompt

In [7]:
zero_shot_prompt = """You are a sentiment classifier.
Classify the sentiment of the text as Positive or Negative.
Answer with only one word: Positive or Negative.

Text: {t}
Answer:"""
results, df = pipeline(valid_df, zero_shot_prompt)


Prompt Template:
----------
You are a sentiment classifier.
Classify the sentiment of the text as Positive or Negative.
Answer with only one word: Positive or Negative.

Text: {t}
Answer:
----------
Accuracy: 87.71%
F1-Score: 0.8839499189056447
--------------------------------------------------


Few-shot prompt

In [14]:
# Few-shot prompt with 3 examples
train_df = pd.read_csv('data/6/train.csv')
train_examples = train_df.sample(5, random_state=2501676)
# Distribution of labels in validation set should be similar to examples
valid_df['sentiment'].value_counts(normalize=True)

sentiment
positive    0.795714
negative    0.204286
Name: proportion, dtype: float64

In [23]:
positive_sample = train_df[train_df['sentiment'] == 'positive'].sample(2, random_state=2501676)
clean_povitive_examples = clean_text(positive_sample)

negative_sample = train_df[train_df['sentiment'] == 'negative'].sample(1, random_state=2501676)
clean_negative_examples = clean_text(negative_sample)

print("Positive Examples:")
print(clean_povitive_examples['text'].tolist()[0])
print(clean_povitive_examples['text'].tolist()[1])
print("\nNegative Example:")
print(clean_negative_examples['text'].tolist()[0])

Positive Examples:
Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
This item worked perfectly for my refrigerator.

Negative Example:
Bought it June 21, died last night. No longevity. Worked good other then that.


In [25]:
one_shot_prompt = """You are a sentiment classifier.
Answer with only one word: Positive or Negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: Positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(valid_df, one_shot_prompt)
results_df = pd.concat([results_df, pd.DataFrame([results_one_shot])], ignore_index=True)

Prompt Template:
----------
You are a sentiment classifier.
Answer with only one word: Positive or Negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: Positive

Text: {t}
Answer:
----------
Accuracy: 90.14%
F1-Score: 0.9072445243901734
--------------------------------------------------


In [26]:
two_shot_prompt = """You are a sentiment classifier.
Answer with only one word: Positive or Negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: Positive

Text: Bought it June 21, died last night. No longevity. Worked good other then that.
Answer: Negative

Text: {t}
Answer:"""

results_two_shot, df = pipeline(valid_df, two_shot_prompt)
results_df = pd.concat([results_df, pd.DataFrame([results_two_shot])], ignore_index=True)

Prompt Template:
----------
You are a sentiment classifier.
Answer with only one word: Positive or Negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: Positive

Text: Bought it June 21, died last night. No longevity. Worked good other then that.
Answer: Negative

Text: {t}
Answer:
----------
Accuracy: 86.71%
F1-Score: 0.8772892986578518
--------------------------------------------------


In [24]:
few_shot_prompt = """You are a sentiment classifier.
Answer with only one word: Positive or Negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: Positive

Text: This item worked perfectly for my refrigerator.
Answer: Positive

Text: Bought it June 21, died last night. No longevity. Worked good other then that.
Answer: Negative

Text: {t}
Answer:"""

results_few, df = pipeline(valid_df, few_shot_prompt)
results_df = pd.concat([results_df, pd.DataFrame([results_few])], ignore_index=True)

Prompt Template:
----------
You are a sentiment classifier.
Answer with only one word: Positive or Negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: Positive

Text: This item worked perfectly for my refrigerator.
Answer: Positive

Text: Bought it June 21, died last night. No longevity. Worked good other then that.
Answer: Negative

Text: {t}
Answer:
----------
Accuracy: 86.29%
F1-Score: 0.873510633243753
--------------------------------------------------


Label-definition prompt

In [30]:
lable_definition_prompt = """Classify the sentiment of the text.

Positive = happy, satisfied, favorable
Negative = unhappy, dissatisfied, unfavorable

Answer with only one word: Positive or Negative.

Text: {t}
Answer:"""
results_lable_definition, df = pipeline(valid_df, lable_definition_prompt)
results_df = pd.concat([results_df, pd.DataFrame([results_lable_definition])], ignore_index=True)

Prompt Template:
----------
Classify the sentiment of the text.

Positive = happy, satisfied, favorable
Negative = unhappy, dissatisfied, unfavorable

Answer with only one word: Positive or Negative.

Text: {t}
Answer:
----------
Accuracy: 85.71%
F1-Score: 0.8670711527854384
--------------------------------------------------


Different Examples in one shot

In [27]:
for exp in train_df['text'].tail(10):
    one_shot_prompt = f"""You are a sentiment classifier.
    Answer with only one word: Positive or Negative.

    Text: {exp}
    Answer: Positive

    """+"""Text: {t}
    Answer:"""
    print(exp)

    results_one_shot, df = pipeline(train_df, one_shot_prompt)


With a second person to help tilt the machine back, all four were quickly swapped-out from the bottom.  I was amazed at how quickly it was done, but they've made just the difference we wanted!  Decent steel and plastic construction, so I'm hoping these last at least as long as the OE ones.
F1-Score: 0.9160581689985677
--------------------------------------------------
It worked out perfectly
F1-Score: 0.9075777943739108
--------------------------------------------------
Don't bother... Within a year flashs an EEO code and no longer functions. Customer service is a terminal hold.
F1-Score: 0.9082042660502259
--------------------------------------------------
They were affordable and easy to install- look better than the original.
F1-Score: 0.9167043607066729
--------------------------------------------------
It was an exact fit at a good price.
F1-Score: 0.9075777943739108
--------------------------------------------------
Plug'n and play!!!
F1-Score: 0.9121290322580644
----------------

In [28]:
one_shot_prompt = """You are a sentiment classifier.
Answer with only one word: Positive or Negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: Positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 0.9213053613053613
--------------------------------------------------
